In [0]:
bronze = "abfss://bronze@databricktraveljournal.dfs.core.windows.net"
table = "bio"
bronze_table_parquet_path = f"{bronze}/{table}"

bio_df = spark.read.format("parquet")\
    .load(f"{bronze_table_parquet_path}")

display(bio_df)

bio_text,created_at,flag,id,user_id,date_type,year,month,day,_rescued_data
Them institution yard book night religious color.,null,false,419,258,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T06:34:02.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/bio/year=2024/month=8/day=6/part-00002-tid-6676060932920920630-ac9cf053-3c26-46e6-9369-6528b9202d65-1009-2.c000.snappy.parquet""}"
Word approach skin expect husband party team.,null,false,420,259,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T06:08:49.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/bio/year=2024/month=8/day=6/part-00002-tid-6676060932920920630-ac9cf053-3c26-46e6-9369-6528b9202d65-1009-2.c000.snappy.parquet""}"
Whom avoid skill notice.,null,false,421,260,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T14:59:43.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/bio/year=2024/month=8/day=6/part-00002-tid-6676060932920920630-ac9cf053-3c26-46e6-9369-6528b9202d65-1009-2.c000.snappy.parquet""}"
Size civil record sit ask.,null,false,423,262,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T00:00:18.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/bio/year=2024/month=8/day=6/part-00002-tid-6676060932920920630-ac9cf053-3c26-46e6-9369-6528b9202d65-1009-2.c000.snappy.parquet""}"
Everybody bar audience child fund popular. Especially something yourself why project.,null,false,424,263,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T04:03:14.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/bio/year=2024/month=8/day=6/part-00002-tid-6676060932920920630-ac9cf053-3c26-46e6-9369-6528b9202d65-1009-2.c000.snappy.parquet""}"
Total majority investment cover.,null,false,426,265,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T02:14:34.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/bio/year=2024/month=8/day=6/part-00002-tid-6676060932920920630-ac9cf053-3c26-46e6-9369-6528b9202d65-1009-2.c000.snappy.parquet""}"
Year minute though likely but catch Mr.,null,false,427,266,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T09:41:36.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/bio/year=2024/month=8/day=6/part-00002-tid-6676060932920920630-ac9cf053-3c26-46e6-9369-6528b9202d65-1009-2.c000.snappy.parquet""}"
Never analysis how third note art term health.,null,false,430,90,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T07:49:27.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/bio/year=2024/month=8/day=6/part-00002-tid-6676060932920920630-ac9cf053-3c26-46e6-9369-6528b9202d65-1009-2.c000.snappy.parquet""}"
Couple ability here reduce future ok throughout fall. Focus our care face court government listen.,null,false,433,124,2024-08-06,2024,8,6,"{""created_at"":""2024-08-06T01:40:47.000Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/bio/year=2024/month=8/day=6/part-00002-tid-6676060932920920630-ac9cf053-3c26-46e6-9369-6528b9202d65-1009-2.c000.snappy.parquet""}"
KG like to travelloka,null,false,4,14,2026-06-23,2026,6,23,"{""created_at"":""2026-06-23T05:11:33.653Z"",""_file_path"":""abfss://travelsource@databricktraveljournal.dfs.core.windows.net/bio/year=2026/month=6/day=23/part-00000-tid-6676060932920920630-ac9cf053-3c26-46e6-9369-6528b9202d65-1007-8.c000.snappy.parquet""}"


### Quality 

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, coalesce
from pyspark.sql.functions import col, coalesce, get_json_object, to_timestamp, lit, when


def transform_to_silver(bronze_df: DataFrame) -> DataFrame:
    df = bronze_df
                # date_type is a real date; created_at is "-" so we skip it
    df = df.withColumn("created_at",
                               when(col("created_at").isNull(),
                                    get_json_object(col("_rescued_data"),"$.created_at"))
                                    .otherwise(col("created_at"))
                                    )
    df = (
        df
        # if the year is 2024, replace it with 2026 (keeps month/day/time exactly)
        .withColumn(
            "created_at",
            F.when(
                F.col("created_at").startswith("2024"),
                F.regexp_replace("created_at", r"^2024", "2026")
            ).otherwise(F.col("created_at"))
        )
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("date_type", F.to_date("created_at"))
    )

    df = df.withColumn(
            "created_at",
            F.when(
                F.year("created_at") == 2024,
                F.col("created_at") + F.expr("INTERVAL 2 YEARS")
            ).otherwise(F.col("created_at"))
        )

    # date_type is a real date; created_at is "-" so we skip it
    df = (
        df.withColumn("created_at", F.to_timestamp("created_at"))
          .withColumn("date_type", F.to_date("date_type"))
    )


    df = df.withColumn("flag", F.lower(F.trim(F.col("flag"))) == F.lit("true"))

    df = (
        df.withColumn("year",          F.expr("try_cast(year as int)"))
          .withColumn("month",         F.expr("try_cast(month as int)"))
          .withColumn("day",           F.expr("try_cast(day as int)"))
          .withColumn("id",            F.expr("try_cast(id as int)"))
          .withColumn("user_id", F.expr("try_cast(user_id as int)"))
          .withColumn("year",  F.when(F.col("year") == 2024, F.lit(2026)).otherwise(F.col("year")))

    )


    quality_df = df \
            .withColumn("_is_valid", coalesce(
                (col("bio_text").isNotNull()) &
                (col("id").isNotNull()) &
                (col("user_id").isNotNull()) &
                lit(True)
            ))
        

        # Separate valid and invalid records
    valid_df = quality_df.filter(col("_is_valid"))
    invalid_df = quality_df.filter(~col("_is_valid"))

        # Log invalid records for investigation
    if invalid_df.count() > 0:

            print(f"Quarantined {invalid_df.count()} invalid records")


    return valid_df.drop("_rescued_data","_is_valid")


df = transform_to_silver(bio_df)


In [0]:
df.display()

bio_text,created_at,flag,id,user_id,date_type,year,month,day
Them institution yard book night religious color.,2026-08-06T06:34:02Z,false,419,258,2026-08-06,2026,8,6
Word approach skin expect husband party team.,2026-08-06T06:08:49Z,false,420,259,2026-08-06,2026,8,6
Whom avoid skill notice.,2026-08-06T14:59:43Z,false,421,260,2026-08-06,2026,8,6
Size civil record sit ask.,2026-08-06T00:00:18Z,false,423,262,2026-08-06,2026,8,6
Everybody bar audience child fund popular. Especially something yourself why project.,2026-08-06T04:03:14Z,false,424,263,2026-08-06,2026,8,6
Total majority investment cover.,2026-08-06T02:14:34Z,false,426,265,2026-08-06,2026,8,6
Year minute though likely but catch Mr.,2026-08-06T09:41:36Z,false,427,266,2026-08-06,2026,8,6
Never analysis how third note art term health.,2026-08-06T07:49:27Z,false,430,90,2026-08-06,2026,8,6
Couple ability here reduce future ok throughout fall. Focus our care face court government listen.,2026-08-06T01:40:47Z,false,433,124,2026-08-06,2026,8,6
KG like to travelloka,2026-06-23T05:11:33.653Z,false,4,14,2026-06-23,2026,6,23


### Deduplicated


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def deduplicate_by_key(df, keycolumns, order_column, ascending=False):
    """
    
    Deduplicate a Dataframe by composite key, keeping the row with the highest (or lowest) value in order_column

    Args:
        df: Input DataFrame with duplicates
        key_columns: List of columns forming the composite key
        order_column: Column to break ties (e.g., updated_at)
        ascending: If True, keep the smallest order_column value
    """
    
    order_expr = (
        F.col(order_column).asc() if ascending else F.col(order_column).desc()
    )

    window_spec = Window.partitionBy(*keycolumns).orderBy(order_expr)

    return df.withColumn("rank", F.row_number().over(window_spec)).filter(
        F.col("rank") == 1
    ).drop("rank")

bio_df = deduplicate_by_key(df, ["id"], "created_at", ascending=True)



In [0]:
bio_df.display()

bio_text,created_at,flag,id,user_id,date_type,year,month,day
112223,2026-06-17T22:39:29.032Z,false,1,8,2026-06-17,2026,6,17
pureun kub,2026-06-21T19:45:02.971Z,false,2,10,2026-06-21,2026,6,21
Anything else?,2026-06-22T18:23:28.633Z,false,3,13,2026-06-22,2026,6,22
KG like to travelloka,2026-06-23T05:11:33.653Z,false,4,14,2026-06-23,2026,6,23
Pat パット 🧑‍💻 🇹🇭 🇯🇵 CU ME 🇹🇭 | KYUTECH 🇯🇵 [🤖⚙️🧑‍💻] Building the best version. 🏆 一期一会 ⏳️ 🧗 @pat.tiny.steps 🎬 TT: patto.life,2026-06-23T21:01:47.669Z,false,5,16,2026-06-23,2026,6,23
Dream,2026-06-25T18:36:28.899Z,false,6,17,2026-06-25,2026,6,25
Several behind decision laugh fire. Age oil action heart.,2026-07-10T04:33:23.64Z,false,40,7,2026-07-10,2026,7,10
Identify agree me because design generation claim.,2026-06-27T14:33:23.772Z,false,41,16,2026-06-27,2026,6,27
In stop southern would. Simple operation simply mouth.,2026-07-13T12:48:45.412Z,false,58,18,2026-07-13,2026,7,13
Hit effect the.,2026-07-06T07:48:45.674Z,false,60,6,2026-07-06,2026,7,6


### Data Writing 

In [0]:
bio_df.write.format("delta").mode("overwrite").save("abfss://silver@databricktraveljournal.dfs.core.windows.net/bio")

# DELTA

In [0]:
%sql

CREATE TABLE IF NOT EXISTS travel_journal_catalog.silver.bio 
USING DELTA
LOCATION "abfss://silver@databricktraveljournal.dfs.core.windows.net/bio"

In [0]:
%sql
SELECT * FROM travel_journal_catalog.silver.bio

bio_text,created_at,flag,id,user_id,date_type,year,month,day
112223,2026-06-17T22:39:29.032Z,false,1,8,2026-06-17,2026,6,17
pureun kub,2026-06-21T19:45:02.971Z,false,2,10,2026-06-21,2026,6,21
Anything else?,2026-06-22T18:23:28.633Z,false,3,13,2026-06-22,2026,6,22
KG like to travelloka,2026-06-23T05:11:33.653Z,false,4,14,2026-06-23,2026,6,23
Pat パット 🧑‍💻 🇹🇭 🇯🇵 CU ME 🇹🇭 | KYUTECH 🇯🇵 [🤖⚙️🧑‍💻] Building the best version. 🏆 一期一会 ⏳️ 🧗 @pat.tiny.steps 🎬 TT: patto.life,2026-06-23T21:01:47.669Z,false,5,16,2026-06-23,2026,6,23
Dream,2026-06-25T18:36:28.899Z,false,6,17,2026-06-25,2026,6,25
Several behind decision laugh fire. Age oil action heart.,2026-07-10T04:33:23.64Z,false,40,7,2026-07-10,2026,7,10
Identify agree me because design generation claim.,2026-06-27T14:33:23.772Z,false,41,16,2026-06-27,2026,6,27
In stop southern would. Simple operation simply mouth.,2026-07-13T12:48:45.412Z,false,58,18,2026-07-13,2026,7,13
Hit effect the.,2026-07-06T07:48:45.674Z,false,60,6,2026-07-06,2026,7,6
